# 🚗 YOLOv11 Car Accident Detection — End-to-End Training Pipeline**Project:** Harnessing Deep Learning to Optimize Emergency Response  **Model:** YOLOv11 (Ultralytics) — Classification  **Dataset:** CADP (Car Accident Detection and Prediction)  ---## Pipeline Overview1. **Setup** — Install dependencies, mount Drive, configure paths, import libraries2. **Dataset Exploration** — Explore CADP structure & annotations3. **Dataset Preparation** — Label frames, filter with vehicle detector, split train/val/test4. **Model Training** — Train YOLOv11-cls with checkpoint management5. **Evaluation** — Metrics, confusion matrix, sample predictions6. **Inference** — Run on new video clips7. **Export** — ONNX export for deployment> ⚠️ **GPU Required:** Go to `Runtime > Change runtime type > T4 GPU`### 💾 Smart CachingLong-running cells save their results to Google Drive. After a runtime restart,  they detect previous work and skip automatically — no repeated computation.

---## 1. Setup & EnvironmentRun cells 1.1 and 1.2 in order. If 1.1 installs new packages, **restart the runtime**  (Runtime → Restart session), then **skip 1.1** and start from **1.2**.

In [ ]:
# ============================================================
# 1.1 Install Dependencies (run ONCE per session)
# ============================================================
# After running this cell for the first time:
#   1. Runtime > Restart session
#   2. Skip this cell
#   3. Continue from cell 1.2
# ============================================================
!pip install -q ultralytics albumentations scikit-learn pyyaml tqdm opencv-python-headless

print("✅ Dependencies installed.")
print("⚠️  If this is your first run: Runtime > Restart session, then skip to 1.2")

In [ ]:
# ============================================================
# 1.2 Mount Drive + Imports + Paths (run this FIRST after restart)
# ============================================================
# This single cell handles everything so you never get NameErrors
# ============================================================

# --- Mount Google Drive ---
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("✅ Drive already mounted")

# --- All Imports ---
import json
import glob
import shutil
import random
import pickle
import time
import numpy as np
import cv2
import torch
import yaml
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from IPython.display import display, Image as IPImage
from ultralytics import YOLO

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Source Data (CADP dataset) ---
CADP_ROOT       = Path('/content/drive/MyDrive/Car_accident_detection')
CADP_DATASET    = CADP_ROOT / 'Dataset'
CADP_MANUAL     = CADP_DATASET / 'manual'
FRAMES_DIR      = CADP_MANUAL / 'extracted_frames'   # THE CORE DATA
ANNO_FILE       = CADP_MANUAL / 'annotations_1531762138.1303267.json'

# --- Working Directory (persistent on Drive) ---
WORK_DIR        = Path('/content/drive/MyDrive/Colab Notebooks/grad-project/program-memory')
CACHE_DIR       = WORK_DIR / 'cache'           # cached intermediate results
CHECKPOINT_DIR  = WORK_DIR / 'checkpoints'
RUNS_DIR        = WORK_DIR / 'runs'

# --- Classification Dataset (prepared data on Drive) ---
DATASET_DIR     = WORK_DIR / 'cls_dataset'     # train/val/test with accident/no_accident

# --- Create directories ---
for d in [CACHE_DIR, CHECKPOINT_DIR, RUNS_DIR, DATASET_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Verify GPU ---
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🖥️  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("❌ No GPU! Go to Runtime > Change runtime type > T4 GPU")

# --- Verify dataset ---
if FRAMES_DIR.exists():
    clip_folders = sorted([d for d in FRAMES_DIR.iterdir() if d.is_dir()])
    print(f"📂 Found {len(clip_folders)} clips in extracted_frames/")
else:
    print(f"❌ Frames not found at {FRAMES_DIR}")
    clip_folders = []

print(f"\n✅ All imports, paths, and GPU ready")
print(f"   CADP Frames:  {FRAMES_DIR}")
print(f"   Dataset Dir:  {DATASET_DIR}")
print(f"   Cache Dir:    {CACHE_DIR}")
print(f"   Runs Dir:     {RUNS_DIR}")

---## 2. Explore CADP DatasetThe CADP dataset has **pre-extracted frames** in `manual/extracted_frames/`.  Each subfolder (e.g., `000909/`) is one video clip with individual `frame_XXXXXX.jpg` files.**Labeling convention** (from the CADP paper):- Frames **0–90** → Normal traffic (`no_accident`)- Frames **91–104** → Ambiguous boundary → **skip**- Frames **105+** → Accident (`accident`)

In [ ]:
# ============================================================
# 2.1 Explore Clip Structure
# ============================================================
if clip_folders:
    print(f"📂 {len(clip_folders)} clip folders in extracted_frames/\n")

    # Show first 5 clips
    for folder in clip_folders[:5]:
        frames = sorted(folder.glob('*.jpg'))
        print(f"  📁 {folder.name}/ → {len(frames)} frames", end="")
        if frames:
            print(f"  (first: {frames[0].name}, last: {frames[-1].name})")
        else:
            print()

    if len(clip_folders) > 5:
        print(f"  ... and {len(clip_folders) - 5} more clips")

    # Stats
    all_frame_counts = [len(list(f.glob('*.jpg'))) for f in clip_folders]
    print(f"\n📊 Frame count stats:")
    print(f"   Min: {min(all_frame_counts)}, Max: {max(all_frame_counts)}, "
          f"Avg: {np.mean(all_frame_counts):.0f}, Total: {sum(all_frame_counts)}")
else:
    print("❌ No clip folders found")

In [ ]:
# ============================================================
# 2.2 Visualize Sample Frames from a Clip
# ============================================================
if clip_folders:
    sample_clip = clip_folders[0]
    frames = sorted(sample_clip.glob('*.jpg'))

    print(f"📂 Clip: {sample_clip.name} ({len(frames)} frames)\n")

    # Show start (normal), middle (ambiguous zone), and end (accident)
    indices = [0, min(50, len(frames)-1), min(90, len(frames)-1),
               min(105, len(frames)-1), len(frames)-1]
    labels = ["Frame 0\n(Normal)", "Frame 50\n(Normal)", "Frame 90\n(Normal boundary)",
              "Frame 105\n(Accident start)", f"Frame {len(frames)-1}\n(Late accident)"]

    fig, axes = plt.subplots(1, min(5, len(indices)), figsize=(20, 4))
    for ax, idx, label in zip(axes, indices, labels):
        if idx < len(frames):
            img = cv2.imread(str(frames[idx]))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(label, fontsize=10)
        ax.axis('off')

    plt.suptitle(f"Clip {sample_clip.name} — Frame Timeline", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 2.3 Explore Annotation File
# ============================================================
if ANNO_FILE.exists():
    with open(ANNO_FILE, 'r') as f:
        raw_annotations = json.load(f)
    print(f"✅ Annotation file loaded: {len(raw_annotations)} entries")
    print(f"   Keys are YouTube video IDs (e.g., {list(raw_annotations.keys())[:3]})")
    print(f"\n   Note: These map to videos in forth_investigation/, not extracted_frames/")
    print(f"   For extracted_frames/ we use the CADP paper convention:")
    print(f"   frames 0-90 = normal, 91-104 = skip, 105+ = accident")
else:
    print("⚠️  No annotation file found (expected for extracted_frames/ workflow)")
    print("   Using CADP paper convention: frames 0-90 = normal, 105+ = accident")

---## 3. Build Classification DatasetThis section:1. Labels frames using the CADP convention (0–90 = normal, 105+ = accident)2. Filters normal frames through YOLOv8n to ensure vehicles are visible3. Splits into train/val/test4. **Saves to Google Drive** so it never needs to run again💾 **Smart caching**: If the dataset already exists on Drive, this cell loads it instantly.

In [ ]:
# ============================================================
# 3.1 Build Dataset with Vehicle Filter (RESUMABLE)
# ============================================================
# Processes clip-by-clip and saves progress to Drive.
# If the session dies, re-run this cell — it picks up where
# it left off. No work is repeated.
# ============================================================

ACCIDENT_START  = 105
AMBIGUOUS_START = 91
TRAIN_RATIO     = 0.70
VAL_RATIO       = 0.20

VEHICLE_CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
MIN_CONFIDENCE  = 0.3

PROGRESS_FILE = CACHE_DIR / 'build_progress.json'
CACHE_FILE    = CACHE_DIR / 'dataset_build_summary.json'

def load_progress():
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {'completed_clips': [], 'saved': {}, 'skipped': {}}

def save_progress(progress):
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(progress, f)

def dataset_already_built():
    if not CACHE_FILE.exists():
        return False
    for split in ['train', 'val', 'test']:
        for label in ['accident', 'no_accident']:
            d = DATASET_DIR / split / label
            if not d.exists() or len(list(d.glob('*.jpg'))) == 0:
                return False
    return True

if dataset_already_built():
    with open(CACHE_FILE) as f:
        summary = json.load(f)
    print("⚡ Dataset already on Drive! Skipping.\n")
    for split in ['train', 'val', 'test']:
        for label in ['accident', 'no_accident']:
            count = len(list((DATASET_DIR / split / label).glob('*.jpg')))
            print(f"   {split}/{label}: {count} frames")
    print(f"\n💡 To rebuild: delete {DATASET_DIR} and {PROGRESS_FILE}")

else:
    # Create folders
    for split in ['train', 'val', 'test']:
        for label in ['accident', 'no_accident']:
            (DATASET_DIR / split / label).mkdir(parents=True, exist_ok=True)

    # Deterministic clip order + split assignment
    clips = sorted([d for d in FRAMES_DIR.iterdir() if d.is_dir()])
    random.Random(SEED).shuffle(clips)

    n_train = int(len(clips) * TRAIN_RATIO)
    n_val   = int(len(clips) * VAL_RATIO)

    clip_split = {}
    for i, c in enumerate(clips):
        if i < n_train:
            clip_split[c.name] = 'train'
        elif i < n_train + n_val:
            clip_split[c.name] = 'val'
        else:
            clip_split[c.name] = 'test'

    # Load progress from any previous partial run
    progress = load_progress()
    done_set = set(progress['completed_clips'])
    remaining = [c for c in clips if c.name not in done_set]

    if done_set:
        print(f"🔄 Resuming! {len(done_set)} clips done, {len(remaining)} remaining.\n")
    else:
        print(f"🔧 Starting fresh — {len(clips)} clips to process.\n")

    # Load filter model
    print("Loading YOLOv8n for vehicle filtering...")
    filter_model = YOLO("yolov8n.pt")
    print("✅ Filter model ready\n")

    saved   = progress.get('saved', {s: {'accident': 0, 'no_accident': 0} for s in ['train', 'val', 'test']})
    skipped = progress.get('skipped', {'no_vehicle': 0, 'ambiguous': 0})

    # Ensure keys exist (in case of old partial progress)
    for s in ['train', 'val', 'test']:
        saved.setdefault(s, {'accident': 0, 'no_accident': 0})
    skipped.setdefault('no_vehicle', 0)
    skipped.setdefault('ambiguous', 0)

    for clip_dir in tqdm(remaining, desc="Building dataset"):
        split  = clip_split[clip_dir.name]
        frames = sorted(clip_dir.glob('*.jpg'), key=lambda p: int(p.stem))

        for frame_path in frames:
            frame_num = int(frame_path.stem)

            if frame_num < AMBIGUOUS_START:
                # Normal candidate — run vehicle filter
                results = filter_model.predict(str(frame_path), verbose=False, conf=MIN_CONFIDENCE)
                detected = results[0].boxes.cls.tolist() if results[0].boxes else []
                if not any(int(c) in VEHICLE_CLASSES for c in detected):
                    skipped['no_vehicle'] += 1
                    continue
                label = 'no_accident'

            elif frame_num >= ACCIDENT_START:
                label = 'accident'

            else:
                skipped['ambiguous'] += 1
                continue

            dst = DATASET_DIR / split / label / f"{clip_dir.name}_f{frame_num:05d}.jpg"
            shutil.copy2(str(frame_path), str(dst))
            saved[split][label] += 1

        # Mark clip as done and save progress after EVERY clip
        progress['completed_clips'].append(clip_dir.name)
        progress['saved'] = saved
        progress['skipped'] = skipped
        save_progress(progress)

    # All done — write final summary
    summary = {'total_clips': len(clips), 'saved': saved, 'skipped': skipped}
    with open(CACHE_FILE, 'w') as f:
        json.dump(summary, f, indent=2)

    # Clean up progress file
    PROGRESS_FILE.unlink(missing_ok=True)

    print("\n✅ Dataset complete and saved to Drive!\n")
    print("📊 Summary:")
    for split in ['train', 'val', 'test']:
        for label in ['accident', 'no_accident']:
            print(f"   {split}/{label}: {saved[split][label]} frames")
    print(f"\n🗑️  Skipped:")
    print(f"   No vehicle: {skipped['no_vehicle']}")
    print(f"   Ambiguous:  {skipped['ambiguous']}")

🔄 Resuming! 607 clips done, 809 remaining.

Loading YOLOv8n for vehicle filtering...
✅ Filter model ready



Building dataset:   0%|          | 0/809 [00:00<?, ?it/s]

error: OpenCV(4.13.0) /io/opencv/modules/imgcodecs/src/loadsave.cpp:1310: error: (-215:Assertion failed) !buf.empty() in function 'imdecode_'


In [ ]:
# ============================================================
# 3.2 Visualize Dataset Samples
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Dataset Samples', fontsize=16, fontweight='bold')

for row, label in enumerate(['no_accident', 'accident']):
    samples = list((DATASET_DIR / 'train' / label).glob('*.jpg'))
    if samples:
        chosen = random.sample(samples, min(4, len(samples)))
        for col, img_path in enumerate(chosen):
            ax = axes[row][col]
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            color = 'red' if label == 'accident' else 'green'
            ax.set_title(f"{label}\n{img_path.name}", fontsize=8, color=color)
            ax.axis('off')

plt.tight_layout()
plt.show()

# Class balance
for split in ['train', 'val', 'test']:
    acc = len(list((DATASET_DIR / split / 'accident').glob('*.jpg')))
    norm = len(list((DATASET_DIR / split / 'no_accident').glob('*.jpg')))
    total = acc + norm
    if total > 0:
        print(f"📊 {split}: {acc} accident ({acc/total*100:.0f}%) | "
              f"{norm} no_accident ({norm/total*100:.0f}%) | total: {total}")

In [ ]:
# ============================================================
# 3.3 Create data.yaml
# ============================================================

# For YOLO classification, data.yaml just points to the root folder
# with train/val/test subdirectories containing class folders
DATA_YAML_PATH = DATASET_DIR / 'data.yaml'

data_yaml = {
    'path': str(DATASET_DIR),
    'train': 'train',
    'val': 'val',
    'test': 'test',
    'nc': 2,
    'names': ['no_accident', 'accident']
}

with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f"✅ data.yaml created at: {DATA_YAML_PATH}")
print()
with open(DATA_YAML_PATH) as f:
    print(f.read())

---## 4. YOLOv11 Model TrainingWe use **YOLOv11s-cls** (small classification model) for accident vs no_accident classification.Features:- **Checkpoint management** — saves to Drive, resumes after disconnections- **Early stopping** — prevents overfitting- **Cosine LR schedule** — smooth learning rate decay

In [ ]:
# ============================================================
# 4.1 Training Configuration
# ============================================================

MODEL_VARIANT = 'yolo11s-cls.pt'   # Classification model (small)

TRAINING_CONFIG = {
    'data':       str(DATASET_DIR),
    'epochs':     100,
    'batch':      32,            # Classification can use larger batch
    'imgsz':      224,           # Standard classification size (faster than 640)
    'patience':   20,            # Early stopping
    'save_period': 5,            # Checkpoint every 5 epochs
    'device':     0,
    'workers':    2,
    'project':    str(RUNS_DIR),
    'name':       'accident_cls_yolov11',
    'exist_ok':   True,
    'pretrained':  True,
    'optimizer':  'AdamW',
    'lr0':        0.001,
    'lrf':        0.01,
    'cos_lr':     True,
    'warmup_epochs': 3,
    'seed':       SEED,

    # Augmentation (conservative for CCTV footage)
    'hsv_h': 0.015,
    'hsv_s': 0.5,
    'hsv_v': 0.3,
    'degrees': 3.0,        # Slight rotation only
    'translate': 0.1,
    'scale': 0.3,
    'flipud': 0.0,         # No vertical flip for CCTV
    'fliplr': 0.5,
    'erasing': 0.1,
}

print("✅ Training configuration ready")
print(f"   Model:      {MODEL_VARIANT}")
print(f"   Epochs:     {TRAINING_CONFIG['epochs']} (patience={TRAINING_CONFIG['patience']})")
print(f"   Batch:      {TRAINING_CONFIG['batch']}")
print(f"   Image size: {TRAINING_CONFIG['imgsz']}")
print(f"   Output:     {RUNS_DIR / 'accident_cls_yolov11'}")

In [ ]:
# ============================================================
# 4.2 Train (with Auto-Resume Support)
# ============================================================
# Set RESUME = True if training was interrupted by a disconnect.
# The model will pick up from the last saved checkpoint.
# ============================================================

RESUME = False  # ← Set True after a disconnect

RUN_DIR = RUNS_DIR / 'accident_cls_yolov11'

if RESUME:
    last_pt = RUN_DIR / 'weights' / 'last.pt'
    if last_pt.exists():
        print(f"🔄 Resuming from: {last_pt}")
        model = YOLO(str(last_pt))
        results = model.train(resume=True)
    else:
        print("⚠️  No last.pt found — starting fresh")
        RESUME = False

if not RESUME:
    print(f"🚀 Starting fresh training with {MODEL_VARIANT}\n")
    model = YOLO(MODEL_VARIANT)
    results = model.train(**TRAINING_CONFIG)

# Backup best weights to checkpoint dir
best_pt = RUN_DIR / 'weights' / 'best.pt'
if best_pt.exists():
    backup_path = CHECKPOINT_DIR / 'accident_cls_best.pt'
    shutil.copy2(str(best_pt), str(backup_path))
    print(f"\n💾 Best weights backed up to: {backup_path}")

print("\n✅ Training complete!")

---## 5. Evaluation & Metrics

In [ ]:
# ============================================================
# 5.1 Validate on Test Set
# ============================================================

# Load best model
best_pt = CHECKPOINT_DIR / 'accident_cls_best.pt'
if not best_pt.exists():
    best_pt = RUN_DIR / 'weights' / 'best.pt'

if best_pt.exists():
    eval_model = YOLO(str(best_pt))
    print(f"✅ Loaded best model: {best_pt.name}")
else:
    eval_model = model
    print("⚠️  Using last trained model (best.pt not found)")

# Run validation on test split
test_results = eval_model.val(
    data=str(DATASET_DIR),
    split='test',
    batch=32,
    imgsz=224,
    verbose=True
)

print("\n" + "=" * 50)
print("📊 TEST SET RESULTS")
print("=" * 50)
print(f"  Top-1 Accuracy: {test_results.top1:.4f}")
print(f"  Top-5 Accuracy: {test_results.top5:.4f}")
print("=" * 50)

In [ ]:
# ============================================================
# 5.2 Display Training Curves
# ============================================================

RUN_DIR = RUNS_DIR / 'accident_cls_yolov11'

plot_files = ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png']

for plot_name in plot_files:
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        print(f"\n📈 {plot_name}")
        display(IPImage(filename=str(plot_path), width=800))
    else:
        print(f"⚠️  {plot_name} not found")

In [ ]:
# ============================================================
# 5.3 Inference on Sample Test Images
# ============================================================

test_accident = list((DATASET_DIR / 'test' / 'accident').glob('*.jpg'))
test_normal   = list((DATASET_DIR / 'test' / 'no_accident').glob('*.jpg'))

# Mix accident and normal samples
samples = []
if test_accident:
    samples += random.sample(test_accident, min(4, len(test_accident)))
if test_normal:
    samples += random.sample(test_normal, min(4, len(test_normal)))
random.shuffle(samples)

n_show = min(8, len(samples))
if n_show == 0:
    print("❌ No test images found")
else:
    cols = min(4, n_show)
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    fig.suptitle('Test Set Predictions', fontsize=16, fontweight='bold')

    for idx in range(len(axes)):
        ax = axes[idx]
        if idx < n_show:
            img_path = samples[idx]

            # Get ground truth from folder name
            gt_label = img_path.parent.name

            # Run prediction
            pred = eval_model.predict(str(img_path), verbose=False)
            pred_class = pred[0].probs.top1
            pred_name  = pred[0].names[pred_class]
            pred_conf  = pred[0].probs.top1conf.item()

            correct = (pred_name == gt_label)

            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)

            color = 'green' if correct else 'red'
            symbol = '✅' if correct else '❌'
            ax.set_title(f"{symbol} GT: {gt_label}\nPred: {pred_name} ({pred_conf:.0%})",
                        fontsize=9, color=color)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

---## 6. Inference on Video ClipsRun the trained classifier on full video clips — classifies each frame as accident / no_accident.

In [ ]:
# ============================================================
# 6.1 Run Inference on a Video
# ============================================================

def classify_video(model, video_path, conf_threshold=0.5, max_frames=500, sample_every=3):
    """
    Run frame-by-frame classification on a video.
    Returns timeline of predictions.
    """
    video_path = Path(video_path)
    if not video_path.exists():
        print(f"❌ Video not found: {video_path}")
        return None

    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"🎬 {video_path.name}: {total} frames @ {fps:.0f}fps ({total/fps:.1f}s)")

    timeline = []
    frame_idx = 0

    for _ in tqdm(range(min(total, max_frames)), desc="Classifying"):
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % sample_every == 0:
            # Save temp frame for prediction
            temp_path = '/content/temp_frame.jpg'
            cv2.imwrite(temp_path, frame)

            pred = model.predict(temp_path, verbose=False)
            pred_class = pred[0].probs.top1
            pred_name  = pred[0].names[pred_class]
            pred_conf  = pred[0].probs.top1conf.item()

            timeline.append({
                'frame': frame_idx,
                'time_sec': frame_idx / fps,
                'prediction': pred_name,
                'confidence': pred_conf
            })

        frame_idx += 1

    cap.release()

    # Find first accident detection
    accidents = [t for t in timeline if t['prediction'] == 'accident' and t['confidence'] >= conf_threshold]

    print(f"\n✅ Processed {frame_idx} frames")
    print(f"   Accident detections: {len(accidents)} / {len(timeline)} sampled frames")
    if accidents:
        first = accidents[0]
        peak  = max(accidents, key=lambda x: x['confidence'])
        print(f"   First accident at: {first['time_sec']:.1f}s (conf: {first['confidence']:.0%})")
        print(f"   Peak confidence:   {peak['confidence']:.0%} at {peak['time_sec']:.1f}s")

    return timeline

# --- Run on a sample video from CADP ---
# Check forth_investigation for videos
video_dir = CADP_DATASET / 'forth_investigation'
if video_dir.exists():
    test_videos = sorted(video_dir.glob('*.mp4'))[:1]
    if test_videos:
        timeline = classify_video(eval_model, test_videos[0])

        # Plot timeline
        if timeline:
            times = [t['time_sec'] for t in timeline]
            is_acc = [1 if t['prediction'] == 'accident' else 0 for t in timeline]
            confs  = [t['confidence'] for t in timeline]

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
            ax1.plot(times, is_acc, 'r-', linewidth=1.5)
            ax1.fill_between(times, is_acc, alpha=0.3, color='red')
            ax1.set_ylabel('Accident Detected')
            ax1.set_title(f'Accident Detection Timeline — {test_videos[0].name}')

            ax2.plot(times, confs, 'b-', linewidth=1)
            ax2.set_ylabel('Confidence')
            ax2.set_xlabel('Time (seconds)')

            plt.tight_layout()
            plt.show()
    else:
        print("No .mp4 files found in forth_investigation/")
else:
    print("forth_investigation/ not found — upload a video to test")

---## 7. Export & Summary

In [ ]:
# ============================================================
# 7.1 Export Model to ONNX
# ============================================================

best_pt = CHECKPOINT_DIR / 'accident_cls_best.pt'
if not best_pt.exists():
    best_pt = RUN_DIR / 'weights' / 'best.pt'

if best_pt.exists():
    export_model = YOLO(str(best_pt))

    onnx_path = export_model.export(format='onnx', imgsz=224, simplify=True)
    print(f"\n✅ ONNX exported: {onnx_path}")

    # Backup to Drive
    if Path(onnx_path).exists():
        dest = CHECKPOINT_DIR / 'accident_cls_best.onnx'
        shutil.copy2(onnx_path, str(dest))
        print(f"💾 Backed up to: {dest}")
else:
    print("❌ No best weights found. Train the model first (Section 4).")

In [ ]:
# ============================================================
# 7.2 Summary Report
# ============================================================

# Count dataset
counts = {}
for split in ['train', 'val', 'test']:
    counts[split] = {}
    for label in ['accident', 'no_accident']:
        counts[split][label] = len(list((DATASET_DIR / split / label).glob('*.jpg')))

total_images = sum(c for s in counts.values() for c in s.values())

print("=" * 60)
print("📋 TRAINING SUMMARY")
print("=" * 60)
print()
print(f"  Project:        Accident Detection (YOLOv11)")
print(f"  Task:           Classification (accident vs no_accident)")
print(f"  Model:          {MODEL_VARIANT}")
print(f"  Dataset:        CADP — manual/extracted_frames/")
print(f"  Total Images:   {total_images}")
print()
for split in ['train', 'val', 'test']:
    acc  = counts[split]['accident']
    norm = counts[split]['no_accident']
    print(f"  {split:>5}: {acc + norm} images (accident={acc}, normal={norm})")
print()
print(f"  📂 Files on Drive:")
print(f"     Dataset:     {DATASET_DIR}")
print(f"     Best model:  {CHECKPOINT_DIR / 'accident_cls_best.pt'}")
print(f"     ONNX:        {CHECKPOINT_DIR / 'accident_cls_best.onnx'}")
print(f"     Runs:        {RUNS_DIR}")
print()
print("=" * 60)
print()
print("🎓 Next Steps:")
print("  1. Review confusion matrix — check for systematic errors")
print("  2. Try larger model (yolo11m-cls.pt) if accuracy is low")
print("  3. Add more data from outsource/ and Lijun/ folders")
print("  4. Try imgsz=640 for better small-object handling")
print("  5. Integrate into your Flask/FastAPI backend")